In [296]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import joblib
import json
import os

In [280]:
# Train/test
feature = pd.read_csv("D:/credit-card-crosssell/data/feature_store_historical.csv")
label   = pd.read_csv("D:/credit-card-crosssell/data/label_historical.csv")

df = feature.merge(label[['customer_id', 'converted']], on='customer_id')

drop_cols = ['customer_id', 'observation_date']
X = df.drop(columns=drop_cols + ['converted'])
y = df['converted']

# OOT
oot_feature = pd.read_csv("D:/credit-card-crosssell/data/feature_store_oot.csv")
oot_label   = pd.read_csv("D:/credit-card-crosssell/data/label_oot.csv")

oot = oot_feature.merge(oot_label[['customer_id', 'converted']], on='customer_id')

X_oot = oot.drop(columns=drop_cols + ['converted'])
y_oot = oot['converted']

print(f"X shape     : {X.shape}")
print(f"y shape     : {y.shape}")
print(f"X_oot shape : {X_oot.shape}")
print(f"y_oot shape : {y_oot.shape}")

X shape     : (10000, 54)
y shape     : (10000,)
X_oot shape : (3000, 54)
y_oot shape : (3000,)


In [281]:
category = ['has_loan', 'has_term_deposit', 'has_savings_account']
numeric = ['age', 'gender', 'province', 'tenure_months', 'has_mobile_banking', 'num_products', 'has_loan',
       'salary_mean_12m', 'salary_std_12m', 'salary_cv_12m', 'salary_min_12m',
       'salary_max_12m', 'salary_trend_12m', 'salary_delay_rate_12m',
       'has_term_deposit', 'term_deposit_count', 'term_deposit_total',
       'term_deposit_interest_annual', 'has_savings_account',
       'savings_avg_balance_12m', 'savings_min_balance_avg',
       'savings_trend_12m', 'total_income_proxy', 'bills_electric_avg',
       'bills_water_avg', 'bills_internet_avg', 'bills_total_avg',
       'bills_to_income_ratio', 'edu_spend_avg', 'entertainment_avg',
       'grocery_spend_avg', 'dining_spend_avg', 'shopping_spend_avg',
       'transport_spend_avg', 'total_shopping_avg', 'transfer_avg',
       'total_spend_avg', 'discretionary_spend_avg', 'essential_spend_ratio',
       'spend_to_income_ratio', 'monthly_surplus_avg', 'cashflow_volatility',
       'months_negative_balance', 'overdraft_rate', 'high_spend_months',
       'end_month_shortfall_avg', 'spending_acceleration', 'savings_rate_avg',
       'wealth_accumulation', 'edu_spend_trend', 'large_transfer_count',
       'large_transfer_avg', 'income_jump_detected', 'lifestyle_inflation']

In [282]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, train_size=0.7, random_state=42, stratify=y)

try mutal info and rfecv

In [283]:
# mutual info
X_encoded = X_train.copy()

# Encode categorical columns
cat_cols = ['gender', 'province']
for col in cat_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])

X_encoded['random_noise'] = np.random.normal(0, 1, len(X_encoded))

mi_scores = mutual_info_classif(X_encoded, y_train, random_state=42)
mi_df     = pd.DataFrame({
    'feature' : X_encoded.columns,
    'mi_score': mi_scores
})

# Ngưỡng = MI của random feature
noise_threshold = mi_df[mi_df['feature'] == 'random_noise']['mi_score'].values[0]
selected = mi_df[mi_df['mi_score'] > noise_threshold]['feature'].tolist()

mi_df = pd.DataFrame({
    'feature' : X_encoded.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

mi_df[mi_df['feature'] == 'random_noise']

,feature,mi_score
54,random_noise,0.0


data to synethic. try rfecv

In [284]:
# Step 0: Encode categoricals 
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()
X_oot_encoded = X_oot.copy()

In [285]:
# Fit encoder trên train
encoders = {}
cat_cols = ['gender', 'province']

for col in cat_cols:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train[col])
    encoders[col] = le  # lưu lại

for col in cat_cols:
    X_test_encoded[col] = encoders[col].transform(X_test[col])
    X_oot_encoded[col]  = encoders[col].transform(X_oot[col])

In [286]:
# Step 1: Correlation filter 
corr_matrix = X_train_encoded.corr().abs()

# Upper triangle
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Drop feature có correlation > 0.7 với feature khác
to_drop = [col for col in upper.columns if any(upper[col] > 0.7)]
X_filtered = X_train_encoded.drop(columns=to_drop)

print(f"Before correlation filter : {X_train_encoded.shape[1]} features")
print(f"Dropped                   : {len(to_drop)} features → {to_drop}")
print(f"After correlation filter  : {X_filtered.shape[1]} features")

Before correlation filter : 54 features
Dropped                   : 23 features → ['salary_std_12m', 'salary_min_12m', 'salary_max_12m', 'term_deposit_count', 'term_deposit_total', 'term_deposit_interest_annual', 'savings_avg_balance_12m', 'savings_min_balance_avg', 'total_income_proxy', 'bills_total_avg', 'bills_to_income_ratio', 'total_shopping_avg', 'total_spend_avg', 'discretionary_spend_avg', 'essential_spend_ratio', 'spend_to_income_ratio', 'monthly_surplus_avg', 'cashflow_volatility', 'overdraft_rate', 'end_month_shortfall_avg', 'savings_rate_avg', 'wealth_accumulation', 'large_transfer_avg']
After correlation filter  : 31 features


In [287]:
# Step 2: RFECV 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rfecv = RFECV(
    estimator=lgb.LGBMClassifier(
        n_estimators=100,
        random_state=42,
        verbose=-1
    ),
    step=2,
    cv=cv,
    scoring='roc_auc',
    min_features_to_select=5,
    n_jobs=-1
)

rfecv.fit(X_filtered, y_train)

,estimator,"LGBMClassifie...2, verbose=-1)"
,step,2
,min_features_to_select,5
,cv,StratifiedKFo... shuffle=True)
,scoring,'roc_auc'
,verbose,0
,n_jobs,-1
,importance_getter,'auto'
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1


In [288]:

selected = X_filtered.columns[rfecv.support_].tolist()
dropped  = X_filtered.columns[~rfecv.support_].tolist()

print(f"\nOptimal features : {rfecv.n_features_}")
print(f"Selected         : {selected}")
print(f"Dropped by RFECV : {dropped}")


Optimal features : 27
Selected         : ['age', 'gender', 'tenure_months', 'has_mobile_banking', 'num_products', 'salary_mean_12m', 'salary_cv_12m', 'salary_trend_12m', 'salary_delay_rate_12m', 'has_savings_account', 'savings_trend_12m', 'bills_electric_avg', 'bills_water_avg', 'bills_internet_avg', 'edu_spend_avg', 'entertainment_avg', 'grocery_spend_avg', 'dining_spend_avg', 'shopping_spend_avg', 'transport_spend_avg', 'transfer_avg', 'months_negative_balance', 'high_spend_months', 'spending_acceleration', 'edu_spend_trend', 'income_jump_detected', 'lifestyle_inflation']
Dropped by RFECV : ['province', 'has_loan', 'has_term_deposit', 'large_transfer_count']


In [289]:
from xgboost import XGBClassifier

In [290]:
# train model
model = XGBClassifier()

model.fit(X_train_encoded[selected],y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [291]:
from sklearn.metrics import roc_auc_score

In [292]:
y_train_preds = model.predict_proba(X_train_encoded[selected])
auc_train = roc_auc_score(y_train,y_train_preds[:,1])

In [293]:
y_oot_preds = model.predict_proba(X_oot_encoded[selected])
auc_oot = roc_auc_score(y_oot,y_oot_preds[:,1])

auc_oot

0.8953596771186347

overfit -> optuna for tuning

In [294]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 50, 500),
        'max_depth'         : trial.suggest_int('max_depth', 2, 6),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 10, 100),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'gamma'             : trial.suggest_float('gamma', 0.0, 5.0),
        'random_state'      : 42,
        'eval_metric'       : 'auc',
        'verbosity'         : 0,
    }

    model = XGBClassifier(**params)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model,
        X_train_encoded[selected],
        y_train,
        cv      = cv,
        scoring = 'roc_auc',
        n_jobs  = -1
    )

    return scores.mean()


# Chạy tuning
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest AUC CV  : {study.best_value:.4f}")
print(f"Best params  : {study.best_params}")

# Train model tốt nhất
best_model = XGBClassifier(
    **study.best_params,
    random_state = 42,
    eval_metric  = 'auc',
    verbosity    = 0
)
best_model.fit(X_train_encoded[selected], y_train)

# Evaluate
auc_train = roc_auc_score(y_train, best_model.predict_proba(X_train_encoded[selected])[:,1])
auc_test  = roc_auc_score(y_test,  best_model.predict_proba(X_test_encoded[selected])[:,1])
auc_oot   = roc_auc_score(y_oot,   best_model.predict_proba(X_oot_encoded[selected])[:,1])

print(f"\nAUC Train              : {auc_train:.4f}")
print(f"AUC Test               : {auc_test:.4f}")
print(f"AUC OOT                : {auc_oot:.4f}")
print(f"Degradation Train→Test : {auc_train - auc_test:.4f}")
print(f"Degradation Test→OOT   : {auc_test  - auc_oot:.4f}")

  0%|          | 0/50 [00:00<?, ?it/s]


Best AUC CV  : 0.8993
Best params  : {'n_estimators': 165, 'max_depth': 5, 'learning_rate': 0.05729512437313371, 'subsample': 0.8959489892671941, 'colsample_bytree': 0.7826499869105897, 'min_child_weight': 31, 'reg_alpha': 1.2087369320015409, 'reg_lambda': 1.0959982913110289, 'gamma': 4.607734582768523}

AUC Train              : 0.9341
AUC Test               : 0.9033
AUC OOT                : 0.9042
Degradation Train→Test : 0.0308
Degradation Test→OOT   : -0.0009


In [297]:
# Tạo folder models nếu chưa có
os.makedirs("D:/credit-card-crosssell/models", exist_ok=True)

In [298]:
# Lưu model
joblib.dump(best_model, "D:/credit-card-crosssell/models/xgb_propensity.pkl")
print("Saved: xgb_propensity.pkl")

# Lưu best params
with open("D:/credit-card-crosssell/models/best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)
print("Saved: best_params.json")

# Lưu selected features
with open("D:/credit-card-crosssell/models/selected_features.json", "w") as f:
    json.dump(selected, f, indent=2)
print("Saved: selected_features.json")

# Lưu encoders
joblib.dump(encoders, "D:/credit-card-crosssell/models/encoders.pkl")
print("Saved: encoders.pkl")

# Lưu AUC results
results = {
    "auc_train"           : round(auc_train, 4),
    "auc_test"            : round(auc_test, 4),
    "auc_oot"             : round(auc_oot, 4),
    "degradation_train_test": round(auc_train - auc_test, 4),
    "degradation_test_oot"  : round(auc_test  - auc_oot, 4),
}

Saved: xgb_propensity.pkl
Saved: best_params.json
Saved: selected_features.json
Saved: encoders.pkl


In [299]:
with open("D:/credit-card-crosssell/models/model_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved: model_results.json")

print("\nAll files saved to models/")
print(json.dumps(results, indent=2))

Saved: model_results.json

All files saved to models/
{
  "auc_train": 0.9341,
  "auc_test": 0.9033,
  "auc_oot": 0.9042,
  "degradation_train_test": 0.0308,
  "degradation_test_oot": -0.0009
}


In [295]:
# # Check feature nào đang drive AUC cao
# from xgboost import XGBClassifier
# import matplotlib.pyplot as plt

# model = XGBClassifier(n_estimators=100, random_state=42)
# model.fit(X_train_encoded, y_train)

# # Feature importance
# fi = pd.DataFrame({
#     'feature'   : X_train_encoded.columns,
#     'importance': model.feature_importances_
# }).sort_values('importance', ascending=False).head(10)

# print(fi)